In [25]:
from __future__ import print_function, division   # Ensures Python3 printing & division standard
import pandas as pd 
import csv
from pandas import Series, DataFrame 
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import colors
from matplotlib.colors import LogNorm

from sklearn import tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_curve, auc
import lightgbm as lgb 
import xgboost as xgb





In [17]:
#Supervised training
data_class = pd.DataFrame(np.genfromtxt('../InitialProject/Data/AppML_InitialProject_test_classification.csv',
                                  names=True, delimiter=','))
data_regre = pd.DataFrame(np.genfromtxt('../InitialProject/Data/AppML_InitialProject_test_regression.csv',
                                  names=True, delimiter=','))
data_train_full = pd.DataFrame(np.genfromtxt('../InitialProject/Data/AppML_InitialProject_train.csv',
                                  names=True, delimiter=','))
data_truth = data_train_full[['p_Truth_isElectron']]
data_train = data_train_full.drop(columns=['p_Truth_isElectron'])
#Unsupervised training, and not related to the first 3 datasets
data_clust = pd.DataFrame(np.genfromtxt('../InitialProject/Data/SDSS-Gaia_5950stars.csv',
                                  names=True, delimiter=',', 
                                        usecols=range(1, 21)))

# Classification XGBoost
XGBoost was faster and slightly more accurate than LightGBM when the sample size was over  ~3*10**5
I have 1.8*10**5 for training. Alomst same accuracy, but XGBoost much faster

In [23]:

# We train 3 times to get variation/uncertainty:
# X_train = data_frame.drop('isn', axis=1)
# y_train = data_frame['isn']

# Performance and time for LGB
# data_train = lgb.Dataset(X_train, label=y_train)
params = {'objective': 'binary',
    'boosting_type': 'gbdt',
    'metric': 'binary_logloss',
    'learning_rate': 0.02,
    'num_leaves': 10,
    'max_depth': 10,
    'min_data': 10,
    'verbose': 0,
    'force_col_wise': True}

# lgb_model = lgb.train(params, data_train, 500)
# accuracy_lgb_temp.append(accuracy_score(y_test, lgb_model.predict(X_test) > 0.5))

# Performance and time for XGB
xgb_model = xgb.XGBClassifier(objective="binary:logistic", random_state=42)

xgb_model.fit(data_train, data_truth)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [24]:
accuracy_score(data_truth, xgb_model.predict(data_train) > 0.5)


0.9950888888888889

In [ ]:


# Get probability scores for the positive class
y_prob = xgb_model.predict_proba(data_class)[:, 1]

# You need true labels to plot ROC curve
# If you have them:
y_true = data_truth  # your true labels

# Calculate ROC curve
fpr, tpr, thresholds = roc_curve(y_true, y_prob)

# Calculate AUC score
roc_auc = auc(fpr, tpr)

# Plot the ROC curve
plt.figure(figsize=(10, 8))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc="lower right")
plt.show()

ValueError: feature_names mismatch: ['averageInteractionsPerCrossing', 'p_Rhad1', 'p_Rhad', 'p_f3', 'p_weta2', 'p_Rphi', 'p_Reta', 'p_Eratio', 'p_f1', 'p_TRTPID', 'p_numberOfInnermostPixelHits', 'p_numberOfPixelHits', 'p_numberOfSCTHits', 'p_numberOfTRTHits', 'p_TRTTrackOccupancy', 'p_numberOfTRTXenonHits', 'p_z0', 'p_d0', 'p_sigmad0', 'p_dPOverP', 'p_deltaEta1', 'p_deltaPhiRescaled2', 'p_etcone20', 'p_etcone30', 'p_etcone40', 'p_ptcone20', 'p_ptcone30', 'p_ptcone40', 'p_ptPU30', 'p_vertex', 'pX_E7x7_Lr2', 'pX_E7x7_Lr3', 'pX_E_Lr0_HiG', 'pX_E_Lr0_MedG', 'pX_E_Lr1_HiG', 'pX_E_Lr1_LowG', 'pX_E_Lr1_MedG', 'pX_E_Lr2_HiG', 'pX_E_Lr2_LowG', 'pX_E_Lr2_MedG', 'pX_E_Lr3_HiG', 'pX_E_Lr3_MedG', 'pX_MultiLepton', 'pX_OQ', 'pX_ambiguityType', 'pX_asy1', 'pX_author', 'pX_barys1', 'pX_core57cellsEnergyCorrection', 'pX_deltaEta0', 'pX_deltaEta1', 'pX_deltaEta2', 'pX_deltaEta3', 'pX_deltaPhi0', 'pX_deltaPhi1', 'pX_deltaPhi2', 'pX_deltaPhi3', 'pX_deltaPhiFromLastMeasurement', 'pX_deltaPhiRescaled0', 'pX